In [1]:
import pandas as pd
import numpy as np
from astropy.constants import R_sun
import sympy as sp

In [2]:
df_AllParams = pd.read_excel("/mnt/c/Users/luukv/Documenten/NatuurSterrkenkundeMasterProject/CodeMP/MasterProject/tables/Parameters/AllParam.xlsx")
df_FitResults = pd.read_excel("/mnt/c/Users/luukv/Documenten/NatuurSterrkenkundeMasterProject/CodeMP/MasterProject/tables/Parameters/FitResults.xlsx")

df_AllParams = pd.merge(df_AllParams, df_FitResults[['id', 'vsini_model', 'vsini_model_err+', 'vsini_model_err-']], on='id')

#### Calculate the rotational period of the OB star

In [3]:
def POB_error(row):

    # Define the symbols
    ROPT, I, VSINI, SigmaROPT, SigmaI, SigmaVSINI = sp.symbols(
        'ROPT I VSINI SigmaROPT SigmaI SigmaVSINI'
    )

    # Define the function L
    POB = (2 * np.pi * ROPT * (R_sun.value / 1000) * sp.sin(sp.rad(I))) / VSINI / (60 * 60 * 24)

    # Calculate the partial derivatives
    partial_derivative_ROPT = sp.diff(POB, ROPT)
    partial_derivative_I = sp.diff(POB, I)
    partial_derivative_VSINI = sp.diff(POB, VSINI)

    # Calculate the error expression
    error_POB = sp.sqrt(
        (partial_derivative_ROPT * SigmaROPT)**2 +
        (partial_derivative_I * SigmaI)**2 +
        (partial_derivative_VSINI * SigmaVSINI)**2)


    values = {
        ROPT: row['Ropt'],
        I: row['i'],
        VSINI: row['vsini_model'],
        SigmaROPT: row['Ropt_err'],
        SigmaI: row['i_err'],
        SigmaVSINI: row['vsini_model_err+']
    }

    # Calculate the error expression with values substituted
    error_POB_with_values = error_POB.subs(values)

    # Calculate the numerical value
    numerical_value_error_POB = error_POB_with_values.evalf()

    return numerical_value_error_POB

In [4]:
df_AllParams['P_OB'] = (2 * np.pi * df_AllParams['Ropt'] * (R_sun.value / 1000) * np.sin(np.deg2rad(df_AllParams['i']))) / df_AllParams['vsini_model'] / (60 * 60 * 24)
# Apply the function row-wise
df_AllParams['P_OB_err'] = df_AllParams.apply(POB_error, axis=1)

#### Maximun inclination possible

In [5]:
df_AllParams['i_max'] = 90 - np.rad2deg(np.arcsin(df_AllParams['Ropt'] / df_AllParams['a']))

In [6]:
df_AllParams[['id', 'P_OB', 'P_OB_err', 'Porb,ecl', 'Porb,ecl_err', 'vsini_model','vsin(i)', 'i', 'i_max', 'a']].to_excel("/mnt/c/Users/luukv/Documenten/NatuurSterrkenkundeMasterProject/CodeMP/MasterProject/tables/Important/CoRotation.xlsx")